# Part 6: Generative AI for Recommender Explainability

In the evaluation attributes for Problem Statement 2, there is a bonus point available for **"The AI Edge"**: 
*"Extra weight will be given to teams that creatively leverage modern, state-of-the-art Artificial Intelligence (LLMs) to solve complex nuances..."*

In our architecture (`llm_service.py`), we implement a layer 4 transparent AI generation node. Once LightGBM successfully ranks the Cart Add-ons, we asynchronously ask Gemini / Groq: *"Why did we recommend this?"*

In [ ]:
import os
import json

# Mock implementation of our actual prompt mechanics

## 1. The Dynamic Prompt
We don't just ask the LLM to guess; we feed it the *deterministic insights* generated by our Feature Engineering pipeline (Part 2), so it rarely hallucinates.

In [ ]:
def construct_prompt(cart_items, recommended_item, user_context):
    return f"""
    Act as Zomato's friendly food recommendation assistant.
    
    Current Cart: {', '.join(cart_items)}
    Recommended Add-on: {recommended_item}
    Context: {user_context}
    
    In exactly one short, appetizing sentence (max 80 characters), tell the user why this item pairs perfectly with their cart. 
    Be conversational, enticing, and do not use robotic phrasing like 'I recommend this because'.
    """

cart = ["Hyderabadi Chicken Dum Biryani"]
add_on = "Mirchi Ka Salan"
context = "Dinner time, High Item2Vec Similarity"
    
print("Generated Prompt to LLM:\n")
print(construct_prompt(cart, add_on, context))

## 2. LLM Engine Router (Gemini -> Groq)
Because LLM APIs occasionally rate limit or fail, and because this sits in a high-throughput recommendation pipeline, we implemented a cascading fallback router to guarantee high availability.

In [ ]:
def mock_gemini_api(prompt):
    # Simulate an API failure
    raise ConnectionError("Gemini API Rate Limit Exceeded: 429")

def mock_groq_api(prompt):
    return "Spicy Salan perfectly balances the rich flavors of your Dum Biryani."

def get_explanation(prompt):
    try:
        return mock_gemini_api(prompt)
    except Exception as e:
        print(f"[Fallback Triggered] Primary LLM failed: {e}")
        return mock_groq_api(prompt)

print(f"Final Rendered UI Text: \"{get_explanation(construct_prompt(cart, add_on, context))}\"")

## 3. The Caching Layer
Running an LLM call for *every* recommendation request would destroy our <250ms SLA. 

Our solution:
We hash the `set(Cart)` + `Add on` and save the LLM response to Redis with a TTL of 72 hours. 
If another user adds "Biryani" to their cart and we recommend "Salan", the explanation is instantly fetched from Redis in <5ms.

In [ ]:
import hashlib

def generate_cache_key(cart_items, recommendation):
    sorted_cart = ",".join(sorted(cart_items)) # Order-agnostic hash
    combined = f"{sorted_cart}||{recommendation}"
    
    return "llm_expl_" + hashlib.md5(combined.encode()).hexdigest()

key = generate_cache_key(cart, add_on)
print(f"Redis Cache Key: {key}")
print("Value: Spicy Salan perfectly balances the rich flavors of your Dum Biryani.")

## Conclusion
This hybrid architecture allows Zomato to provide "Spotify-like" hyper-personalized text while guaranteeing the latency and throughput requirements of a massive logistics ecosystem.